# Exercise 1 -  Apply anonymization on a dataset and evaluate model impact
We will use data swapping to anonymize the data but keep it in a form which can be processed by ML models.

We use the Adult dataset. Predict whether annual income of an individual exceeds $50K/yr based on census data. - Becker, B. & Kohavi, R. (1996). Adult [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

def get_class_metrics(y_true, y_pred):
    print(f"ROC-AUC: {roc_auc_score(y_true, y_pred)}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred)}")
    print(f"Precision: {precision_score(y_true, y_pred)}")
    print(f"Recall: {recall_score(y_true, y_pred)}")
    print(f"F1: {f1_score(y_true, y_pred)}")

# Load the UCI Adult dataset
def load_data():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    columns = [
        "age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "sex",
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
    ]
    df = pd.read_csv(url, header=None, names=columns, skipinitialspace=True)

    df_nan_ind = df[(df == "?").sum(axis=1) > 0].index
    df.drop(df_nan_ind, inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df

# Apply data swapping on selected columns
def data_swapping(df, columns_to_swap, swap_fraction=0.3, random_state=42):
    df_swapped = df.copy()
    rng = np.random.default_rng(seed=random_state)

    n = len(df)
    num_swaps = int(n * swap_fraction)

    for col in columns_to_swap:
        indices = np.arange(n)
        swap_indices = rng.choice(indices, size=num_swaps, replace=False)
        shuffled_values = df_swapped.loc[swap_indices, col].sample(frac=1.0, random_state=random_state).values
        df_swapped.loc[swap_indices, col] = shuffled_values

    return df_swapped

# Train and evaluate model
def train_and_evaluate(df, label_column="income"):
    X = df.drop(label_column, axis=1)
    y = df[label_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    get_class_metrics(y_test, preds)
    df_fi = pd.DataFrame({"Feature name":clf.feature_names_in_, "Feature importance": clf.feature_importances_}).sort_values("Feature importance", ascending=False)
    display(df_fi.head(5))

def preprocess_data(df):
    _df = df.copy()
    _df.drop(columns=["fnlwgt", "education", "relationship", "native-country"], inplace=True)
    _df = pd.get_dummies(data=_df, columns=["workclass", "marital-status", "sex", "race", "occupation"])
    _df["income"] = _df["income"] == "<=50K"

    return _df


Run data swapping to anonymize the data.

In [ ]:
# Main experiment
df_original = load_data()

# Apply data swapping on quasi-identifiers
columns_to_swap = ["age","workclass", "education-num", "occupation", "sex", "race",]
df_swapped = data_swapping(df_original, columns_to_swap, swap_fraction=0.5)

In [ ]:
df_original.head(5)

In [ ]:
df_swapped.head(5)

Run the classification for both dataset and evaluate the models performance.

In [ ]:
# Clean data
df_orig_pre = preprocess_data(df_original)
df_swap_pre = preprocess_data(df_swapped)

# Train models
print("--- Original Results ---")
acc_orig = train_and_evaluate(df_orig_pre)

print("\n")
print("--- Swapped Results ---")
acc_swap = train_and_evaluate(df_swap_pre)

At the end it is clearly visible that performance of the model which used swapped data is decreased, but not signifinactly. Moreover both of models used the same features based on the feature importances.